# **Steps for Data preprocessing**
1.   Import necessary libraries
2.   Read Dataset
3.   Sanity check of data
4.   Exploratory Data Analysis (EDA)
5.   Missing Value treatments
6.   Outliers treatments
7.   Duplicates & Garbage Value treatment
8.   Normalization
9.   Encoding of the data



# **Step 1: Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## **Step 2: Read Dataset**

**Reading dataset directly from Kaggle**

In [ ]:
# Step 1: Install kagglehub
!pip install kagglehub

In [ ]:
# Step 2: Download the Dataset
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kumarajarshi/life-expectancy-who")

print("Path to dataset files:", path)

In [ ]:
# Step 3: Check Files in the Dataset Folder
import os

print(os.listdir(path))

In [ ]:
# Step 4: Read the CSV File
import pandas as pd
import os

csv_file = os.path.join(path, "Life Expectancy Data.csv")

df = pd.read_csv(csv_file)

df.head()

**Reading dataset from Local zipped file**

In [ ]:
# Step 1: Upload the ZIP file
from google.colab import files
# Upload file manually
uploaded = files.upload()
#Step 2: Extract the ZIP file
import zipfile

zip_file_name = 'whoLifeExpectancy.zip'

with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall('data_folder')


In [ ]:
# Upload file manually (if not zip)
uploaded = files.upload()

In [ ]:
# Step 3: Read the CSV (or other file)
import pandas as pd
df = pd.read_csv('/content/data_folder/Life Expectancy Data.csv')

#top 5 records from the dataset
df.head()

In [ ]:
# bottom 5 records from the dataset
df.tail()

# **Step 3: Sanity check of data**
Note: A sanity check for a dataset is a quick, initial assessment to verify that data is rational, consistent, and free from glaring errors before performing in-depth analysis.

In [ ]:
# Shape (note: shape is an attribute)
df.shape

In [ ]:
#info
df.info()

In [ ]:
#displaying data for Nepal
nepal_data = df[df['Country'] == 'Nepal']
print(nepal_data)

#displaying avg. life expectancy for Nepal
average_life = nepal_data['Life expectancy '].mean()
print(average_life)

In [ ]:
#finding the missing value

df.isnull().sum()

In [ ]:
dfc=df.copy()
dfc.head()

In [ ]:
#finding the missing value with percentage

dfc.isnull().sum()/dfc.shape[0]*100

In [ ]:
#finding duplicates
dfc.duplicated().sum()

In [ ]:
#identify any garbage value in the dataset | check the column with object datatype for any garbage

for i in dfc.select_dtypes(include="object").columns:
  print(dfc[i].value_counts())
  print("***"*10)

# **Step 3: Exploratory Data Analysis (EDA)**

In [ ]:
#stastical description
dfc.describe()

In [ ]:
dfc.describe(include="object")

In [ ]:
#histogram to understand the distribution
import warnings
warnings.filterwarnings("ignore")
for i in dfc.select_dtypes(include="number").columns:
  sns.histplot(data=dfc, x=i)
  plt.show()

In [ ]:
#histogram to understand the outliers
import warnings
warnings.filterwarnings("ignore")
for i in dfc.select_dtypes(include="number").columns:
  sns.boxplot(data=dfc, x=i)
  plt.show()

In [ ]:
#scatter plot to understand the relationship (in this case 'Life expectancy ')

dfc.select_dtypes(include="number").columns

In [ ]:
for i in ['Year', 'Adult Mortality', 'infant deaths',
       'Alcohol', 'percentage expenditure', 'Hepatitis B', 'Measles ', ' BMI ',
       'under-five deaths ', 'Polio', 'Total expenditure', 'Diphtheria ',
       ' HIV/AIDS', 'GDP', 'Population', ' thinness  1-19 years',
       ' thinness 5-9 years', 'Income composition of resources', 'Schooling']:
       sns.scatterplot(data=dfc,x=i,y='Life expectancy ')
       plt.show()

In [ ]:
#correlation with heatmap to interpret the relation and multicollinarity

selected_data=dfc.select_dtypes(include="number").corr()
plt.figure(figsize=(15,15))
sns.heatmap(selected_data, annot=True)

# **Step 5: Missing Value Treatments**

In [ ]:
#Choose the method of imputing missing value
#Options are mean, median, mode or KNNImputer

dfc.isnull().sum()

In [ ]:
#if its a target variable (output), it should be removed hence no missing value treatment is needed
#For outlier treatment, we can use following rules
#Rule: For categorical values (discrete), we can use mode and for numerical value mean or median

for i in ["BMI", "Polio","Income composition of resources"]:
  dfc[i].fillna(dfc[i].median(), inplace=True)

In [ ]:
#deleting records with null values in specific columns
dfc = dfc.dropna(
    subset=["BMI", "Polio", "Income composition of resources"]
)

In [ ]:
#Sometimes, there mighe be leading or trailing spaces in column heading that needs to be removed
dfc.columns = dfc.columns.str.strip()

In [ ]:
#using sklearn to treat missing value

from sklearn.impute import KNNImputer
impute = KNNImputer()


In [ ]:
for i in dfc.select_dtypes(include="number").columns:
  dfc[i]=impute.fit_transform(dfc[[i]])

In [ ]:
#check the
dfc.isnull().sum()

In [ ]:
#other alternatives
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

for col in dfc.select_dtypes(include="number").columns:
    dfc[[col]] = imputer.fit_transform(dfc[[col]])

#strategy could be
# strategy="mean"
# strategy="median"
# strategy="most_frequent"
# strategy="constant"


# **Step 6: Outliers treatment**

In [ ]:
# decide whether to do outlier treatment or not
# only applicable to numerical variables with continuous values

def wisker(col):
  q1, q3=np.percentile(col,[25,75])
  #calculate inter quartile range
  iqr=q3-q1
  lw=q1-1.5*iqr
  uw=q3+1.5*iqr
  return lw, uw

In [ ]:
dfc.columns

In [ ]:
wisker(dfc['GDP'])

In [ ]:
dfc.columns = dfc.columns.str.strip()

for i in ['GDP', 'Total expenditure',
          'thinness  1-19 years',
          'thinness 5-9 years']:

    lw, uw = wisker(dfc[i])

    # cap lower outliers
    dfc[i] = np.where(dfc[i] < lw, lw, dfc[i])

    # cap upper outliers
    dfc[i] = np.where(dfc[i] > uw, uw, dfc[i])

In [ ]:
for i in ['GDP', 'Total expenditure',
          'thinness  1-19 years',
          'thinness 5-9 years']:
          sns.boxplot(dfc[i])
          plt.show()

# **Step 7: Duplicates & Garbage Values treatment**
In this case there are no duplicate or garbage values

If there are any duplicate values, we can drop
For the Garbage values, it can be replaced with mdeian/mode of that column

In [ ]:
# dfc.drop_duplicates()

# **Step 8: Normalization**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler() # Min-Max Normalization (0 to 1 Scaling)

# scaler = StandardScaler() # for Standardization (Z-score Scaling)
# scaler = RobustScaler() # Robust Scaling
# scaler = Normalizer() # Normalizer (Row-wise Normalization)

numeric_cols = dfc.select_dtypes(include="number").columns

dfc[numeric_cols] = scaler.fit_transform(dfc[numeric_cols])

# **Step 9: Encoding**

In [ ]:
#Encoding are applied to object data types to numerical
# two aproaches are there label encoding (for categorical) and one hot encoding with pd.getdummies
dfcd=dfc.copy()
dummy=pd.get_dummies(data=dfcd, columns=["Country","Status"],drop_first=True)

In [ ]:
dummy